# COMET で翻訳の品質を評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_machine_translation/comet.ipynb)

NLP クックブック 26-06 ／ 機械翻訳

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

機械翻訳の評価で現在標準的に使われている、人手評価に基づいて学習されたニューラル評価指標を使いたい。

In [1]:
!pip install -q unbabel-comet "setuptools<81"   # comet が使う torchmetrics は、新しい setuptools で削除された pkg_resources を読み込む

In [2]:
import logging, warnings
warnings.filterwarnings("ignore", module="torchmetrics|pytorch_lightning")   # ライブラリの古い書き方についての警告を隠す
from comet import download_model, load_from_checkpoint
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)   # PyTorch Lightning の動作ログ (INFO) を隠す

model = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))   # 原文・訳文・参照訳を使うモデル (約 2.3GB)

data = [
    {"src": "吾輩は猫である。名前はまだ無い。", "mt": "I am a cat. I don't have a name yet.", "ref": "I am a cat. As yet I have no name."},
    {"src": "吾輩は猫である。名前はまだ無い。", "mt": "I am a dog. My name is not here.", "ref": "I am a cat. As yet I have no name."},
    {"src": "彼は昨日東京に行った。", "mt": "He went to Tokyo yesterday.", "ref": "Yesterday he went to Tokyo."},
]
out = model.predict(data, batch_size=8, gpus=1, progress_bar=False)
for d, s in zip(data, out.scores):
    print(f"{s:.3f}  {d['mt']}")
print("システムスコア:", round(out.system_score, 4))

0.949  I am a cat. I don't have a name yet.
0.768  I am a dog. My name is not here.
0.933  He went to Tokyo yesterday.
システムスコア: 0.8834


In [3]:
# 参照訳なしで評価する品質推定 (QE) モデル。利用規約への同意と Hugging Face へのログインが必要（ライセンスは非商用の CC BY-NC-SA 4.0）
# from huggingface_hub import login; login()
# qe = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))
# print(qe.predict([{"src": "彼は昨日東京に行った。", "mt": "He went to Tokyo yesterday."}], batch_size=8, gpus=1).scores)

- BLEU と違い、原文も入力に使う。意味が合っていれば表現が違っても高く評価され、人手評価との相関が高い。上の例でも、参照訳と言い回しや語順が違う 1 行目と 3 行目は 0.93 以上、意味を取り違えた 2 行目 (dog) は 0.77 になる。
- スコアはモデルごとに尺度が違う。異なる COMET モデルの値を比べてはいけない。モデル名を必ず報告する。
- unbabel-comet は numpy 2、transformers 5 系、huggingface-hub 1 系に対応していないため、インストール時にこれらが古い版（numpy 1 系、transformers 4 系など）に入れ替わり、依存関係の衝突を知らせる赤い ERROR と、衝突するライブラリの長い一覧が表示される。このレシピの実行には影響しない。ほかのセルでエラーが出たらランタイムを再起動する。